# LAB | Ensemble Methods

**Load the data**

In this challenge, we will be working with the same Spaceship Titanic data, like the previous Lab. The data can be found here:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv

Metadata

https://github.com/data-bootcamp-v4/data/blob/main/spaceship_titanic.md

In this Lab, you should try different ensemble methods in order to see if can obtain a better model than before. In order to do a fair comparison, you should perform the same feature scaling, engineering applied in previous Lab.

In [ ]:
#Libraries
import pandas as pd
import numpy as np

from matplotlib import pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression,LinearRegression
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier,AdaBoostClassifier, GradientBoostingClassifier

from sklearn.metrics import classification_report,roc_auc_score

from xgboost import XGBClassifier

In [ ]:
spaceship = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/spaceship_titanic.csv")
spaceship.head()

#### Data inspection

In [ ]:
spaceship.shape

In [ ]:
# check if there is imbalance of class in target variable 
spaceship['Transported'].value_counts(normalize = True)

In [ ]:
spaceship.isna().sum()

In [ ]:
spaceship.dtypes

#### create new column : cabin_deck

In [ ]:
spaceship['Cabin_deck'] = spaceship['Cabin'].str[0]

#### Data Precheck

In [ ]:
# target --> categorical variable 

In [ ]:
# explore  distribution of numeric variables 

sns.histplot(spaceship['Age'],bins=10)
plt.show()
sns.histplot(spaceship['RoomService'],bins=30)
plt.show()
sns.histplot(spaceship['FoodCourt'],bins=30)
plt.show()
sns.histplot(spaceship['ShoppingMall'],bins=30)
plt.show()
sns.histplot(spaceship['Spa'],bins=30)
plt.show()
sns.histplot(spaceship['VRDeck'],bins=30)
plt.show()



# explore correlation among numeric variables to see if there might be redundant features 
# Correlation matrix
corr = spaceship.corr(method='pearson',numeric_only=True)
# Mask upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(10,8))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="coolwarm", center=0, cbar=True
)
plt.title("Correlation Heatmap (Lower Triangle Only)")
plt.show()

In [ ]:
# explore relationship between categorical variables and target (categorical as well)

target = "Transported"   
categorical_feats = spaceship.select_dtypes(include=["object", "category", "bool"]).columns.drop([target,'PassengerId','Name','Cabin'])
print(categorical_feats)

for col in categorical_feats:
    print(f"\n{col}")
    display(pd.crosstab(spaceship[col], spaceship[target], normalize="index") * 100)


#### Perform Train Test Split

In [ ]:
# stardard Split

features_df = spaceship.drop(columns=['Transported','PassengerId','Name','Cabin','Destination'])
target = spaceship['Transported']

X_train,X_test,y_train,y_test = train_test_split(features_df,target,test_size=0.2,random_state= 42)
print(f"X train shape is {X_train.shape}")
print(f"X test shape is {X_test.shape}")
print(f"y train shape is {y_train.shape}")
print(f"y test shape is {y_test.shape}")

#### Data Processing


- winsorization handling outlier

- Create an imputer 

In [ ]:


# in features, seperate numeric columns and categorical columns 
num_cols = X_train.select_dtypes(include=["int64","float64"]).columns
cat_cols = X_train.select_dtypes(include=["object","category","bool"]).columns

# create imputer
num_imputer = SimpleImputer(strategy='median')
cat_imputer = SimpleImputer(strategy='most_frequent')

# train imputer using train data , transform train data 
X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
X_train_imputed = pd.concat([X_train[num_cols],X_train[cat_cols]],axis=1)
X_train_imputed

# use trained imputer , transform test data 
X_test[num_cols] = num_imputer.transform(X_test[num_cols])
X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])
X_test_imputed = pd.concat([X_test[num_cols],X_test[cat_cols]],axis=1)
X_test_imputed

cat_cols

- bucketing numeric features -- optional

- encoding categorical features 

In [ ]:

# create encoder object
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# train encoder using train data, transform train data 
X_train_encoded = encoder.fit_transform(X_train_imputed[cat_cols])   
encoder_cols = encoder.get_feature_names_out()          
X_train_encoded_df = pd.DataFrame(X_train_encoded, columns=encoder_cols, index=X_train_imputed.index)
X_train_final = pd.concat([X_train_encoded_df,X_train_imputed[num_cols]],axis=1)

# use trained encoder , transform test data 
X_test_encoded = encoder.transform(X_test_imputed[cat_cols])   
encoder_cols = encoder.get_feature_names_out()          
X_test_encoded_df = pd.DataFrame(X_test_encoded, columns=encoder_cols, index=X_test_imputed.index)
X_test_final = pd.concat([X_test_encoded_df,X_test_imputed[num_cols]],axis=1)
X_test_final

#### Bivarate analysis - filter more predictive features

In [ ]:
# if gini = 1, accurate prediction, if gini = 0, pure guess 
# threshold = 0.1
def gini(y_true,y_predic):
    roc_auc_score(y_true,y_predic)
    return 2 * roc_auc_score(y_true,y_predic) -1 

In [ ]:
# cabin 
# 0.1387167011923942 > 10%, keep the variable 

m = LogisticRegression()

m.fit(X_train_final[[col for col in X_train_final.columns if col.startswith('Cabin')]], y_train)

pred = m.predict(X_test_final[[col for col in X_test_final.columns if col.startswith('Cabin')]])

gini(y_test,pred)



0.1387167011923942

In [302]:
# other numeric features 
# keep RoomService, Spa, VRDeck
dummies = list(X_train_final[[col for col in X_train_final.columns if col.startswith('Cabin') or col.startswith('Home') or col.startswith('Cryo') or col.startswith('VIP')]].columns)

for col in X_train_final.columns.drop(dummies):
    m = LogisticRegression()
    m.fit(X_train_final[[col]],y_train)
    pred = m.predict(X_test_final[[col]])
    print(m.feature_names_in_)
    print(gini(y_test,pred))

['Age']
0.05155577426259139
['RoomService']
0.258095026443268
['FoodCourt']
-0.06791911719963273
['ShoppingMall']
0.0
['Spa']
0.23488871074848072
['VRDeck']
0.21762452411377353


#### apply different ensemble methods in order to get a better model

In [ ]:
y_train = y_train.astype(int)
y_test = y_test.astype(int)


- logistics regression

In [300]:
log = LogisticRegression(penalty= 'l2',random_state= 42,max_iter=100,solver= 'liblinear')
log.fit(X_train_final,y_train)

pred_log = log.predict(X_test_final)

print(classification_report(y_test,pred_log))

              precision    recall  f1-score   support

           0       0.80      0.76      0.78       861
           1       0.77      0.81      0.79       878

    accuracy                           0.78      1739
   macro avg       0.79      0.78      0.78      1739
weighted avg       0.79      0.78      0.78      1739



- Bagging and Pasting

In [ ]:

baggin_reg = BaggingClassifier(estimator=DecisionTreeClassifier(criterion='gini',max_depth=3,random_state=42),
                              n_estimators=200,
                              max_samples=4000,
                              bootstrap=True)

baggin_reg.fit(X_train_final,y_train)

pred_bagging = baggin_reg.predict(X_test_final)

print(classification_report(y_test,pred_bagging))

- Random Forests

In [ ]:
forest = RandomForestClassifier()

forest.fit(X_train_final,y_train)

pred_forest = forest.predict(X_test_final)

print(classification_report(y_test,pred_forest))

- Gradient Boosting

In [ ]:

gb_classifier = GradientBoostingClassifier(max_depth=10, n_estimators=500, learning_rate=0.05)

gb_classifier.fit(X_train_final,y_train)

pred_gd = gb_classifier.predict(X_test_final)

print(classification_report(y_test,pred_gd))



- xg Boosting

In [ ]:


for c in cat_cols:
    X_train_imputed[c] = X_train_imputed[c].astype("category")
    X_test_imputed[c]  = X_test_imputed[c].astype("category")

xgboost_classifier = XGBClassifier(n_estimators = 300, learning_rate = 0.03, tree_method="hist",enable_categorical=True,random_state=42)
xgboost_classifier.fit(X_train_imputed,y_train)

pred_xg = xgboost_classifier.predict(X_test_imputed)

print(classification_report(y_test,pred_xg))


- Adaptive Boosting

In [ ]:
ada_classfier = AdaBoostClassifier(DecisionTreeClassifier(max_depth=10,random_state=42),
                            n_estimators=100,
                            learning_rate=0.5,random_state=42)

ada_classfier.fit(X_train_final,y_train)

pred_ada = ada_classfier.predict(X_test_final)

print(classification_report(y_test,pred_ada))

Which model is the best and why?

In [ ]:
#comment here